In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Load data
file_path = "Figure_7E_WT_vs_40A_Foci_Number.csv"
df = pd.read_csv(file_path)

# Reshape to long format
df_long = df.melt(var_name='Condition', value_name='Foci Count')
df_long['Construct'] = df_long['Condition'].apply(lambda x: "WT" if "WT" in x else "40A")

# Colors for violins
violin_colors = {"WT": "#49c1bb", "40A": "#e8e8e8"}
condition_order = df.columns.tolist()  # Ensure consistent order

plt.rcParams['font.family'] = 'Arial'
fig, ax = plt.subplots(figsize=(4, 3.2))

# --- Violin plot (full first) ---
vp = sns.violinplot(
    x='Condition', y='Foci Count', data=df_long,
    hue='Construct', order=condition_order,
    inner=None, cut=0, scale='width', palette=violin_colors,
    linewidth=0, dodge=False, ax=ax
)

# --- Clip violins to left half only ---
for violin in ax.collections:
    path = violin.get_paths()[0]
    vertices = path.vertices
    center_x = np.median(vertices[:, 0])  # Find center x
    vertices[:, 0] = np.minimum(vertices[:, 0], center_x)  # Clip to left half
    path.vertices = vertices

# --- Boxplot (white fill) ---
sns.boxplot(
    x='Condition', y='Foci Count', hue='Construct', data=df_long,
    order=condition_order, showcaps=False, showfliers=False, width=0.35,
    boxprops={'facecolor': 'white', 'edgecolor': 'black', 'zorder': 4},
    medianprops={'color': 'black', 'linewidth': 1.2},
    whiskerprops={'color': 'black'}, capprops={'color': 'black'},
    palette=['white'], dodge=False, ax=ax
)

# --- Stripplot (white dots with black edge) ---
sns.stripplot(
    x='Condition', y='Foci Count', hue='Construct', data=df_long,
    order=condition_order, jitter=True, size=3.5, alpha=1,
    palette=['white'], edgecolor='black', linewidth=0.6,
    dodge=False, ax=ax
)

# --- Mean as horizontal line ---
group_means = df_long.groupby('Condition')['Foci Count'].mean()
for i, cond in enumerate(condition_order):
    ax.hlines(
        y=group_means[cond], xmin=i - 0.2, xmax=i + 0.2,
        colors='black', linewidth=1.2, zorder=6
    )

# Remove legends
if ax.get_legend():
    ax.get_legend().remove()

# Vertical group separators
for pos in [1.5, 3.5]:
    ax.axvline(x=pos, color='black', linestyle='--', linewidth=1)

# Labels
plt.xlabel("Condition")
plt.ylabel("# of Foci / cell")
plt.xticks(rotation=45, ha='right')

# Save
plt.savefig("Figure_7E_WT_vs_40A.pdf", format="pdf", dpi=300, bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
import pandas as pd
import scipy.stats as stats
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# --- Load data ---
file_path = "Figure_7E_WT_vs_40A_Foci_Number.csv"
df = pd.read_csv(file_path)

# --- Reshape to long format ---
df_long = df.melt(var_name="Condition", value_name="Foci_Count").dropna()
df_long["Construct"] = df_long["Condition"].apply(lambda x: "WT" if "WT" in x else "40A")
df_long["Source"] = df_long["Condition"].apply(lambda x: "NaCl" if "NaCl" in x else ("KCl" if "KCl" in x else "Sb"))
df_long["Group"] = df_long["Construct"] + "_" + df_long["Source"]

# --- Prepare groups ---
groups = df_long.groupby("Group")["Foci_Count"].apply(list)
pairs = list(combinations(groups.index, 2))

# --- Mann–Whitney U tests ---
results = []
for g1, g2 in pairs:
    u_stat, p_val = stats.mannwhitneyu(groups[g1], groups[g2], alternative='two-sided')
    results.append({"Group1": g1, "Group2": g2, "U_stat": u_stat, "p_uncorrected": p_val})

results_df = pd.DataFrame(results)

# --- FDR correction (Benjamini-Hochberg) ---
reject, p_corrected, _, _ = multipletests(results_df["p_uncorrected"], method='fdr_bh')
results_df["p_corrected_FDR"] = p_corrected
results_df["Significant"] = reject

# --- Sort by corrected p-value ---
results_df = results_df.sort_values("p_corrected_FDR")

# --- Save or display ---
print(results_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Load data
file_path = "Figure_7F_WT_vs_40A_Foci_Intensity.csv"
df = pd.read_csv(file_path)

# Reshape to long format
df_long = df.melt(var_name='Condition', value_name='Foci Count')
df_long['Construct'] = df_long['Condition'].apply(lambda x: "WT" if "WT" in x else "40A")

# Colors for violins
violin_colors = {"WT": "#49c1bb", "40A": "#e8e8e8"}
condition_order = df.columns.tolist()  # Ensure consistent order

plt.rcParams['font.family'] = 'Arial'
fig, ax = plt.subplots(figsize=(4, 3.2))

# --- Violin plot (full first) ---
vp = sns.violinplot(
    x='Condition', y='Foci Count', data=df_long,
    hue='Construct', order=condition_order,
    inner=None, cut=0, scale='width', palette=violin_colors,
    linewidth=0, dodge=False, ax=ax
)

# --- Clip violins to left half only ---
for violin in ax.collections:
    path = violin.get_paths()[0]
    vertices = path.vertices
    center_x = np.median(vertices[:, 0])  # Find center x
    vertices[:, 0] = np.minimum(vertices[:, 0], center_x)  # Clip to left half
    path.vertices = vertices

# --- Boxplot (white fill) ---
sns.boxplot(
    x='Condition', y='Foci Count', hue='Construct', data=df_long,
    order=condition_order, showcaps=False, showfliers=False, width=0.35,
    boxprops={'facecolor': 'white', 'edgecolor': 'black', 'zorder': 4},
    medianprops={'color': 'black', 'linewidth': 1.2},
    whiskerprops={'color': 'black'}, capprops={'color': 'black'},
    palette=['white'], dodge=False, ax=ax
)

# --- Stripplot (white dots with black edge) ---
sns.stripplot(
    x='Condition', y='Foci Count', hue='Construct', data=df_long,
    order=condition_order, jitter=True, size=1, alpha=1,
    palette=['white'], edgecolor='black', linewidth=0.1,
    dodge=False, ax=ax, rasterized=True
)

# --- Mean as horizontal line ---
group_means = df_long.groupby('Condition')['Foci Count'].mean()
for i, cond in enumerate(condition_order):
    ax.hlines(
        y=group_means[cond], xmin=i - 0.2, xmax=i + 0.2,
        colors='black', linewidth=1.2, zorder=6
    )

# Remove legends
if ax.get_legend():
    ax.get_legend().remove()

# Vertical group separators
for pos in [1.5, 3.5]:
    ax.axvline(x=pos, color='black', linestyle='--', linewidth=1)

# Labels
plt.xlabel("Condition")
plt.ylabel("# of Foci / cell")
plt.xticks(rotation=45, ha='right')

# --- Y-axis range ---
ax.set_ylim(top=1000)

# Save
plt.savefig("Figure_7F_WT_vs_40A_Foci_Intensity.pdf", format="pdf", dpi=300, bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
import pandas as pd
import scipy.stats as stats
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# --- Load data ---
file_path = "Figure_7F_WT_vs_40A_Foci_Intensity.csv"
df = pd.read_csv(file_path)

# --- Reshape to long format ---
df_long = df.melt(var_name="Condition", value_name="Foci_Intensity").dropna()
df_long["Construct"] = df_long["Condition"].apply(lambda x: "WT" if "WT" in x else "40A")
df_long["Source"] = df_long["Condition"].apply(lambda x: "NaCl" if "NaCl" in x else ("KCl" if "KCl" in x else "Sb"))
df_long["Group"] = df_long["Construct"] + "_" + df_long["Source"]

# --- Prepare groups ---
groups = df_long.groupby("Group")["Foci_Intensity"].apply(list)
pairs = list(combinations(groups.index, 2))

# --- Mann–Whitney U tests ---
results = []
for g1, g2 in pairs:
    u_stat, p_val = stats.mannwhitneyu(groups[g1], groups[g2], alternative='two-sided')
    results.append({"Group1": g1, "Group2": g2, "U_stat": u_stat, "p_uncorrected": p_val})

results_df = pd.DataFrame(results)

# --- FDR correction (Benjamini-Hochberg) ---
reject, p_corrected, _, _ = multipletests(results_df["p_uncorrected"], method='fdr_bh')
results_df["p_corrected_FDR"] = p_corrected
results_df["Significant"] = reject

# --- Sort by corrected p-value ---
results_df = results_df.sort_values("p_corrected_FDR")

# --- Save or display ---
print(results_df)